In [ ]:
# default_exp stator

In [ ]:
#hide
from nbdev.showdoc import *

In [ ]:
#export
import numpy as np
import matplotlib.pyplot as plt

## stator

> define stator dimensions and calculate its resistance

#### slot resistance

$R_s = \dfrac{\rho n_s^2 L}{kcp A_s } $
$\rho$: resistivity of copper  
$ns$: number of turns per slot  
$L$: stack length  
$kcp$: conductor fill factor  
$A_s$: conductor area   


#### end winding resistance
$R_e = \dfrac{ \rho n_s^2 \pi \tau_c}{2 k_{cp} A_s}$    
$\tau_c = \alpha_{cp} \tau_p$   
$\tau_p = R_{st} \theta_p$  
$\theta_p = \dfrac{2 \pi}{N_p}$  
$N_p$: number of magnet poles   
$R_{st}$: radius of slot top  
$\alpha_{cp}$: coil pitch as a fraction of the pole pitch   



In [ ]:
class Stator:
    def __init__(self, Ns, Np, w_layers, pp, pitch, OD, id_f, bs0_f, hs0, hs1, hs2_f, tw_f, r0, len, turns, slot_fill):
        self.Ns = Ns            # stator slots
        self.Np = Np            # rotor poles
        self.w_layers = w_layers       # winding layers
        self.pp = pp             # parallel paths
        self.pitch = pitch          # slot pitch
        self.OD = OD           # stator outer diameter
        self.id_f = id_f         # stator id fraction
        self.bs0_f = bs0_f        # slot opening fraction
        self.hs0 = hs0            # shoe height
        self.hs1 = hs1          # shoe slope height
        self.hs2_f = hs2_f       # tooth height fraction
        self.tw_f = tw_f         # tooth width as fraction of tooth shoe span
        self.r0 = r0           # radius of curvature
        self.len = len           # stack length
        self.valid = 0          # check valid geometry
        self.turns = turns          # turns per coil
        self.slot_fill = slot_fill    # copper slot fill factor
        self.R_st = 0           # radius of slot top
        self.R_sb = 0           # radius of slot bottom
        self.slot_area = 0      # slot area
        self.phase_res = 0      # phase resistance
    
    def validate_dim(self,):
        """ Validate the given dimensions
        """
        valid_val = 0
    
    def slot_resistance(self,):

        slot_pitch_angle = 2*np.pi/self.Ns
        tooth_angle = slot_pitch_angle*(1-self.bs0_f)
        tw_angle = tooth_angle*self.tw_f
        s_id = self.OD*self.id_f
        # calculate tooth width from the angle made with centre
        tw = s_id*np.sin(tw_angle/2)
        
        # calculate slot bottom radius
        rsb = (s_id/2 + self.hs0 + self.hs1)
        self.R_sb = rsb

        # calculate slot top radius
        rst = rsb + (self.OD/2 - rsb)*self.hs2_f
        self.R_st = rst

        slot_area = np.pi*(rst**2 - rsb**2)/self.Ns - tw*(rst-rsb)
        self.slot_area = slot_area

        ro_cu = 1.72e-8 # copper coefficient of resistance

        r_slot = 1e3*ro_cu*(2*self.len)*(self.turns**2)/(slot_area*self.slot_fill)
        return r_slot

    
    def endwinding_resistance(self, ):
        a_cp = 1                        # coil pitch as factor of slots
        tp = a_cp*self.R_st*np.pi*2/self.Ns  # coil pitch as length
        ro_cu = 1.72e-8                 # copper coefficient of resistance 
        r_endwinding = 1e3*ro_cu*(2*tp)*(self.turns**2)/(self.slot_area*self.slot_fill)
        return r_endwinding
    
    def calc_phase_resistance(self,):
        coils_per_phase = 0.5*self.w_layers*self.Ns/3
        r_coil = (1/self.w_layers)*(self.slot_resistance() + self.endwinding_resistance() )  # coil resistance
        r_phase = r_coil*coils_per_phase/(self.pp**2)                          
        self.phase_res = r_phase

### stator resistance calculation: example

In [ ]:
Ns = 12            # stator slots
Np = 10            # rotor poles
w_layers = 1       # winding layers
pp = 2             # parallel paths
pitch = 1          # slot pitch
OD = 125           # stator outer diameter
id_f = 0.6         # stator id fraction
bs0_f = 0.2        # slot opening fraction
hs0 = 2            # shoe height
hs1 = 0.1          # shoe slope height
hs2_f = 0.72       # tooth height fraction
tw_f = 0.54         # tooth width as fraction of tooth shoe span
r0 = 0.1           # radius of curvature
len = 60           # stack length
turns = 38         # turns per coil
slot_fill = 0.45    # copper slot fill factor

s1 = Stator(Ns, Np, w_layers, pp, pitch, OD, id_f, bs0_f, hs0, hs1, hs2_f, tw_f, r0, len, turns, slot_fill)
s1.calc_phase_resistance()
print('phase resistance: {0:.4f} Ohms'.format(s1.phase_res))

phase resistance: 0.0180 Ohms
